In [1]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

# 1. Load data
df_news = pd.read_csv("data/GOOG_news.csv")
df_news["date"] = pd.to_datetime(df_news["date"]).dt.date

print(f"Articles loaded: {len(df_news)}")
df_news.head()

Articles loaded: 18


,date,title,description
0,2026-03-09,"Dear Oracle Stock Fans, Mark Your Calendars fo...",Oracle stock is heading into a key earnings ev...
1,2026-03-08,Show HN: I logged Gemini's stock predictions f...,Article URL: https://huggingface.co/datasets/l...
2,2026-03-07,Alphabet: Apple AI Deal Is The Biggest Blind S...,Alphabet: Apple AI Deal Is The Biggest Blind S...
3,2026-03-05,Alphabet Stock Edges Lower in Early Trading as...,Alphabet Inc. shares dipped modestly in early ...
4,2026-03-05,Alphabet (GOOGL) is Continuing Its Development...,"Patient Capital Management, a value investing ..."


In [2]:
# 2. Load FinBERT model
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")

finbert = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

print("FinBERT loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded successfully


In [3]:
# 3. Run sentiment analysis
def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return None, None
    result = finbert(text[:512])[0]
    return result["label"], result["score"]

df_news["sentiment"], df_news["confidence"] = zip(
    *df_news["title"].apply(get_sentiment)
)

print(df_news[["date", "title", "sentiment", "confidence"]])

          date                                              title sentiment  \
0   2026-03-09  Dear Oracle Stock Fans, Mark Your Calendars fo...   neutral   
1   2026-03-08  Show HN: I logged Gemini's stock predictions f...   neutral   
2   2026-03-07  Alphabet: Apple AI Deal Is The Biggest Blind S...   neutral   
3   2026-03-05  Alphabet Stock Edges Lower in Early Trading as...  negative   
4   2026-03-05  Alphabet (GOOGL) is Continuing Its Development...  positive   
5   2026-03-03  Alphabet Stock Dips to $306 Amid Geopolitical ...  negative   
6   2026-02-25  IonQ Just Won a Golden Dome Contract. Should Y...   neutral   
7   2026-02-24  Alphabet Stock Holds Steady Near $312 Amid AI ...  positive   
8   2026-02-24  Meta and AMD announce 6-gigawatt GPU deal as p...  positive   
9   2026-02-23  Billionaire David Tepper Just Loaded Up On The...   neutral   
10  2026-02-21  Ray Dalio Sours On America And Sold These Tech...  negative   
11  2026-02-20  Paul Tudor Jones Is Betting Big on G

In [4]:
# 4. Aggregate sentiment by date
# Convert sentiment to a numeric score
sentiment_map = {"positive": 1, "neutral": 0, "negative": -1}
df_news["sentiment_score"] = df_news["sentiment"].map(sentiment_map)

# Average sentiment per day
df_sentiment = df_news.groupby("date").agg(
    sentiment_score=("sentiment_score", "mean"),
    article_count=("title", "count")
).reset_index()

print(f"Days with sentiment data: {len(df_sentiment)}")
print(df_sentiment)

# ── 5. Save ────────────────────────────────────────────────────
df_news.to_csv("data/GOOG_news_sentiment.csv", index=False)
df_sentiment.to_csv("data/GOOG_sentiment_by_day.csv", index=False)

print("Saved successfully")

Days with sentiment data: 13
          date  sentiment_score  article_count
0   2026-02-14              0.0              1
1   2026-02-18              0.0              2
2   2026-02-19              0.0              3
3   2026-02-20              0.0              1
4   2026-02-21             -1.0              1
5   2026-02-23              0.0              1
6   2026-02-24              1.0              2
7   2026-02-25              0.0              1
8   2026-03-03             -1.0              1
9   2026-03-05              0.0              2
10  2026-03-07              0.0              1
11  2026-03-08              0.0              1
12  2026-03-09              0.0              1
Saved successfully
